# Multistage (stage-1 + stage-2) model evaluation

Sweeps every trained multistage checkpoint found under `../models/*/**/*_multistage*/`
(both the plain-MSE and the `_l1` stage-2-loss variants, across every model and
loss strategy), and for each:

1. Computes average metrics (MSE, SSIM, PSNR, LPIPS, FSIM, EPI, TKE, KE, WKE) over
   the full test set (2022-2024), for **both** stage-1 alone and the combined
   `stage1 + epsilon*stage2` model -- so every run directly shows whether stage 2
   actually helped.
2. Plots, for one fixed sample: LR, Stage 1 Pred, the true residual (HR - stage1),
   stage 2's predicted correction (epsilon*stage2), the final combined
   prediction, and HR -- six panels, per channel (u, v).
3. Plots the kinetic-energy spectrum for LR / Stage 1 / Final / HR together, so
   the high-wavenumber gap stage 2 is meant to close is directly visible.

Mirrors `evaluate_all_models.ipynb`'s structure and reuses the same
`evaluate_model()` / `compute_energy_spectrum()` machinery unmodified.

In [13]:
import os
import re
import glob
from collections import OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from torch.utils.data import DataLoader
import sys

sys.path.append('..')

from models.model_factory import create_model
from models.multistage import MultistageModel
from datasets.super_res_dataset import SuperResDataset
from evaluation.evaluate import evaluate_model
from metrics.kinetic_energy_spectrum import compute_energy_spectrum
from train_multistage import load_stage1


In [ ]:
# --- Configuration ---

TEST_FILES = [
    "../hycom_numpy/hycom_2022.npy",
    "../hycom_numpy/hycom_2023.npy",
    "../hycom_numpy/hycom_2024.npy",
]
DOWNSAMPLE_FACTOR = 1
FREQ_THRESHOLD = 10

# Full sweep is slow (batch_size=1 + LPIPS's VGG forward per channel per
# sample, ~6+ min per checkpoint over the full ~974-sample test set), so run
# it one architecture at a time instead of all 5 in one sitting: set this to
# a single model, run the sweep cell below (tqdm shows live progress), then
# come back, change it to the next model, and re-run -- all_results/
# stage1_metrics_cache persist across reruns (see the sweep cell), so each
# pass adds to the same growing result set instead of starting over.
MODELS_TO_TEST = ["SoFoNO"]
BASE_MODEL_DIR = os.path.abspath("../models")

SAMPLE_IDX = 0  # same fixed sample convention as evaluate_all_models.ipynb

# Set QUICK_TEST=True to run the metrics sweep against only the first
# QUICK_TEST_SUBSET_SIZE test samples instead of all ~974 -- useful for
# iterating on the notebook itself or sanity-checking a run without waiting
# on the full thing. The qualitative plots (fixed SAMPLE_IDX) are unaffected
# either way -- they only ever use one sample. Writes to separate CSV/plot
# dirs so a quick run can never be mistaken for, or silently overwrite, a
# real full-sweep result.
QUICK_TEST = False
QUICK_TEST_SUBSET_SIZE = 20

PLOTS_DIR = "eval_plots_multistage_quicktest" if QUICK_TEST else "eval_plots_multistage"
os.makedirs(PLOTS_DIR, exist_ok=True)

RESULTS_CSV = (
    "evaluation_results_multistage_summary_quicktest.csv"
    if QUICK_TEST else "evaluation_results_multistage_summary.csv"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if QUICK_TEST:
    print(f"QUICK_TEST enabled -- metrics sweep will use only {QUICK_TEST_SUBSET_SIZE} test samples")


Using device: cuda


In [15]:
test_dataset = SuperResDataset(
    hr_files=TEST_FILES,
    downsample_factor=DOWNSAMPLE_FACTOR,
    freq_threshold=FREQ_THRESHOLD,
)

# QUICK_TEST only shrinks the loader used for the metrics sweep (evaluate_model
# calls below) -- test_dataset itself stays the full set, since the qualitative
# plots index it directly by SAMPLE_IDX regardless of QUICK_TEST.
if QUICK_TEST:
    from torch.utils.data import Subset
    subset_size = min(QUICK_TEST_SUBSET_SIZE, len(test_dataset))
    metrics_dataset = Subset(test_dataset, range(subset_size))
else:
    metrics_dataset = test_dataset

test_loader = DataLoader(metrics_dataset, batch_size=1, shuffle=False)
print(f"Full test set size: {len(test_dataset)} -- metrics sweep will use: {len(metrics_dataset)}")


Loading datasets...
../hycom_numpy/hycom_2022.npy: (365, 128, 128, 2)
../hycom_numpy/hycom_2023.npy: (365, 128, 128, 2)
../hycom_numpy/hycom_2024.npy: (244, 128, 128, 2)
Loaded 974 samples.
Dataset shape: (974, 128, 128, 2)
Full test set size: 974 -- metrics sweep will use: 974


## Helpers: checkpoint discovery, model loading, plots

In [16]:
def categorize_strategy(experiment_name):
    """Same bucketing as evaluate_all_models.ipynb -- "wke" checked before "ke"
    since "wke" also contains the substring "ke"."""
    name = experiment_name.lower()
    if "wke" in name:
        return "mse_wke"
    if "ke" in name:
        return "mse_lpips_ke"
    if "lpips" in name:
        return "mse_lpips"
    if "mse" in name:
        return "mse"
    return "unknown"


def find_multistage_checkpoints(model_name, base_dir=BASE_MODEL_DIR):
    """Finds every stage-2 best checkpoint under models/{model_name}/*multistage*/,
    labeling each by its base loss strategy and whether stage 2 was trained with
    the "mse" or "l1" STAGE2_LOSS_NORM (train_multistage.py always appends exactly
    one timestamp folder, so the layout here is simpler than the stage-1 one)."""
    model_dir = os.path.join(base_dir, model_name)
    filename = f"{model_name.upper()}_best.pth"
    pattern = os.path.join(model_dir, "*multistage*", "*", "output", "weights", filename)

    found = {}
    for weight_path in sorted(glob.glob(pattern)):
        relative = os.path.relpath(weight_path, model_dir)
        parts = relative.split(os.sep)
        experiment_folder, run_timestamp = parts[0], parts[1]

        loss_norm = "l1" if experiment_folder.endswith("_l1") else "mse"
        base_folder = experiment_folder[:-3] if loss_norm == "l1" else experiment_folder
        base_folder = base_folder.replace("_multistage", "")
        strategy = categorize_strategy(base_folder)

        try:
            run_date = datetime.strptime(run_timestamp, "%d_%m_%Y_%H%M%S")
        except ValueError:
            run_date = datetime.min

        key = (strategy, loss_norm)
        if key not in found or run_date > found[key][0]:
            found[key] = (run_date, weight_path, experiment_folder)

    checkpoints = []
    for (strategy, loss_norm), (run_date, weight_path, experiment_folder) in sorted(found.items()):
        checkpoints.append({
            "strategy": strategy, "loss_norm": loss_norm,
            "weight_path": weight_path, "experiment_folder": experiment_folder,
        })
    return checkpoints


def resolve_from_repo_root(path):
    """STAGE1_CHECKPOINT is stored as a path relative to the repo root (that's
    the cwd train.py/train_multistage.py are run from), but this notebook runs
    with cwd=notebooks/ -- so a plain relative path resolves to the wrong place."""
    if os.path.isabs(path):
        return path
    return os.path.join("..", path)


def load_multistage(model_name, weight_path):
    """Loads stage1 (via its checkpoint path stored inside this checkpoint) and
    stage2, returning (multistage_model, stage1, stage2, epsilon) so callers can
    access the pieces individually for the qualitative panel breakdown."""
    checkpoint = torch.load(weight_path, map_location=device)
    stage1_checkpoint_path = resolve_from_repo_root(checkpoint["stage1_checkpoint"])
    stage1 = load_stage1(model_name.lower(), stage1_checkpoint_path, device)

    stage2 = create_model(model_name.lower(), in_channels=2, out_channels=2).to(device)
    state_dict = checkpoint["model_state_dict"]
    cleaned = OrderedDict(
        (k[len("module."):] if k.startswith("module.") else k, v)
        for k, v in state_dict.items()
    )
    stage2.load_state_dict(cleaned)
    stage2.eval()

    epsilon = checkpoint["epsilon"]
    multistage = MultistageModel(stage1, stage2, epsilon).to(device)
    multistage.eval()
    return multistage, stage1, stage2, epsilon


In [17]:
def plot_stage_comparison(title, lr_t, stage1_pred_t, stage2_pred_t, final_pred_t, hr_t, save_path):
    """LR / Stage1 Pred / True Residual / Stage2 Predicted Correction / Final / HR,
    one row per channel (u, v). Field-like panels (LR, Stage1, Final, HR) share one
    color scale; residual-like panels (true residual, stage2 correction) share a
    separate, much narrower one -- otherwise the residual panels wash out blank
    next to full-magnitude fields."""
    lr_np = lr_t.squeeze(0).cpu().numpy()
    stage1_np = stage1_pred_t.squeeze(0).detach().cpu().numpy()
    final_np = final_pred_t.squeeze(0).detach().cpu().numpy()
    hr_np = hr_t.squeeze(0).cpu().numpy()
    true_residual_np = hr_np - stage1_np
    stage2_correction_np = stage2_pred_t.squeeze(0).detach().cpu().numpy()

    channel_names = ["u-velocity", "v-velocity"]
    fig, axes = plt.subplots(2, 6, figsize=(24, 8))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for ch, ch_name in enumerate(channel_names):
        field_vmin = min(lr_np[ch].min(), stage1_np[ch].min(), final_np[ch].min(), hr_np[ch].min())
        field_vmax = max(lr_np[ch].max(), stage1_np[ch].max(), final_np[ch].max(), hr_np[ch].max())
        res_bound = max(np.abs(true_residual_np[ch]).max(), np.abs(stage2_correction_np[ch]).max())

        panels = [
            ("LR", lr_np[ch], field_vmin, field_vmax),
            ("Stage 1 Pred", stage1_np[ch], field_vmin, field_vmax),
            ("True Residual\n(HR - Stage1)", true_residual_np[ch], -res_bound, res_bound),
            ("Stage 2 Pred\n(error map)", stage2_correction_np[ch], -res_bound, res_bound),
            ("Stage 1 + Stage 2\n(Final)", final_np[ch], field_vmin, field_vmax),
            ("HR (GT)", hr_np[ch], field_vmin, field_vmax),
        ]
        for col, (name, img, vmin, vmax) in enumerate(panels):
            cmap = "RdBu_r" if "Residual" in name or "error map" in name else "viridis"
            im = axes[ch, col].imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
            axes[ch, col].set_title(f"{name} ({ch_name})", fontsize=9)
            axes[ch, col].axis("off")
            plt.colorbar(im, ax=axes[ch, col], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


In [18]:
def plot_ke_spectrum_multistage(title, lr_np, stage1_np, final_np, hr_np, save_path):
    """Same style as evaluate_all_models.ipynb's plot_ke_spectrum, but with an
    extra "Stage 1 Alone" curve, so the plot directly shows whether adding
    stage 2 moved the spectrum closer to HR in the high-wavenumber band."""
    k_lr, E_lr = compute_energy_spectrum(lr_np)
    k_s1, E_s1 = compute_energy_spectrum(stage1_np)
    k_final, E_final = compute_energy_spectrum(final_np)
    k_hr, E_hr = compute_energy_spectrum(hr_np)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), dpi=100, constrained_layout=True)
    fig.suptitle(title, fontsize=13, fontweight="bold")

    ax1.loglog(k_lr, E_lr, label="LR", lw=1.5, color="tab:blue", alpha=0.7)
    ax1.loglog(k_s1, E_s1, label="Stage 1 Alone", lw=2, color="tab:purple", ls="--")
    ax1.loglog(k_final, E_final, label="Stage 1 + Stage 2 (Final)", lw=2, color="tab:orange")
    ax1.loglog(k_hr, E_hr, label="HR", lw=2, color="tab:red")
    ax1.set_title("Complete Spectrum")
    ax1.set_xlabel("Spatial Wavenumber (k)")
    ax1.set_ylabel("Kinetic Energy $E(k)$")
    ax1.grid(True, which="both", ls="--", alpha=0.4)
    ax1.legend(fontsize=8)

    k_max = k_hr.max()
    zoom_lo, zoom_hi = k_max * 0.15, k_max * 0.65
    zoom_mask = (k_hr >= zoom_lo) & (k_hr <= zoom_hi)
    zoom_vals = np.concatenate([E_s1[zoom_mask], E_final[zoom_mask], E_hr[zoom_mask]])
    zoom_vals = zoom_vals[zoom_vals > 0]

    if zoom_mask.sum() >= 2 and zoom_vals.size > 0:
        axins = inset_axes(ax1, width="42%", height="42%", loc="lower left", borderpad=7)
        axins.loglog(k_s1, E_s1, color="tab:purple", lw=2, ls="--")
        axins.loglog(k_final, E_final, color="tab:orange", lw=2)
        axins.loglog(k_hr, E_hr, color="tab:red", lw=2)
        axins.set_xlim(zoom_lo, zoom_hi)
        axins.set_ylim(zoom_vals.min() * 0.5, zoom_vals.max() * 2)
        axins.grid(True, which="both", ls="--", alpha=0.3)
        mark_inset(ax1, axins, loc1=2, loc2=4, fc="none", ec="0.5", lw=1)

    ax2.loglog(k_s1, E_s1, label="Stage 1 Alone", lw=2, color="tab:purple", ls="--")
    ax2.loglog(k_final, E_final, label="Final", lw=2.5, color="tab:orange")
    ax2.loglog(k_hr, E_hr, label="HR", lw=2.5, color="tab:red")
    ax2.set_title("Stage 1 Alone vs Final vs HR")
    ax2.set_xlabel("Spatial Wavenumber (k)")
    ax2.set_ylabel("Kinetic Energy $E(k)$")
    ax2.grid(True, which="both", ls="--", alpha=0.4)
    ax2.legend(fontsize=8)

    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


## Sweep every trained multistage checkpoint

In [19]:
# Persist across reruns: with MODELS_TO_TEST set to one model at a time (see
# the config cell), re-running this cell for the next architecture adds to
# these instead of wiping out the previous architecture's results.
if "all_results" not in dir():
    all_results = {}
if "stage1_metrics_cache" not in dir():
    stage1_metrics_cache = {}  # keyed by stage1 checkpoint path -- shared across mse/l1 variants

for model_name in MODELS_TO_TEST:
    checkpoints = find_multistage_checkpoints(model_name)
    if not checkpoints:
        print(f"Skipping {model_name}: no multistage checkpoints found under {BASE_MODEL_DIR}/{model_name}/")
        continue

    for checkpoint_idx, entry in enumerate(checkpoints, start=1):
        strategy, loss_norm, weight_path = entry["strategy"], entry["loss_norm"], entry["weight_path"]
        run_label = f"{model_name}_{strategy}_{loss_norm}"
        print(f"\n--- Evaluating {run_label} ({checkpoint_idx}/{len(checkpoints)}) ---")
        print(f"Weights: {weight_path}")

        multistage, stage1, stage2, epsilon = load_multistage(model_name, weight_path)
        stage1_checkpoint_path = resolve_from_repo_root(
            torch.load(weight_path, map_location=device)["stage1_checkpoint"]
        )

        # Stage-1-alone metrics: cached, since mse/l1 variants of the same
        # (model, strategy) share the exact same frozen stage-1 checkpoint.
        if stage1_checkpoint_path not in stage1_metrics_cache:
            stage1_metrics_cache[stage1_checkpoint_path] = evaluate_model(
                stage1, test_loader, device, desc=f"{model_name}_{strategy} (stage1)")
        stage1_metrics = stage1_metrics_cache[stage1_checkpoint_path]

        multistage_metrics = evaluate_model(multistage, test_loader, device, desc=f"{run_label} (multistage)")

        all_results[run_label] = {
            "model": model_name, "strategy": strategy, "loss_norm": loss_norm,
            "epsilon": epsilon,
            "stage1": stage1_metrics, "multistage": multistage_metrics,
        }

        # Qualitative plots, fixed sample
        lr, hr = test_dataset[SAMPLE_IDX]
        lr, hr = lr.unsqueeze(0).to(device), hr.unsqueeze(0).to(device)
        with torch.no_grad():
            stage1_pred = stage1(lr)
            stage2_pred = epsilon * stage2(lr)
            final_pred = stage1_pred + stage2_pred

        plot_stage_comparison(
            run_label, lr, stage1_pred, stage2_pred, final_pred, hr,
            save_path=os.path.join(PLOTS_DIR, f"{run_label}_comparison.png"),
        )
        plot_ke_spectrum_multistage(
            run_label,
            lr.squeeze(0).cpu().numpy(),
            stage1_pred.squeeze(0).detach().cpu().numpy(),
            final_pred.squeeze(0).detach().cpu().numpy(),
            hr.squeeze(0).cpu().numpy(),
            save_path=os.path.join(PLOTS_DIR, f"{run_label}_ke_spectrum.png"),
        )

print(f"\n=== Done with {MODELS_TO_TEST}. Total results so far: {len(all_results)} ===")



--- Evaluating SRNO_mse_l1 (1/8) ---
Weights: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRNO/test_composite_loss_mse_multistage_l1/10_09_2026_195526/output/weights/SRNO_best.pth


SRNO_mse_l1 (multistage): 100%|██████████| 974/974 [02:51<00:00,  5.67it/s]



--- Evaluating SRNO_mse_mse (2/8) ---
Weights: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRNO/test_composite_loss_mse_multistage/10_09_2026_180015/output/weights/SRNO_best.pth


SRNO_mse_mse (multistage): 100%|██████████| 974/974 [02:50<00:00,  5.71it/s]
/tmp/ipykernel_40164/2165021957.py:15: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(2, 6, figsize=(24, 8))



--- Evaluating SRNO_mse_lpips_l1 (3/8) ---
Weights: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRNO/test_composite_loss_mse_lpips_0.6_0.4_multistage_l1/10_09_2026_230006/output/weights/SRNO_best.pth


...


--- Evaluating SRNO_mse_wke_l1 (7/8) ---
Weights: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRNO/test_composite_loss_mse_wke_0.6_0.4_multistage_l1/11_09_2026_044249/output/weights/SRNO_best.pth


SRNO_mse_wke_l1 (multistage): 100%|██████████| 974/974 [02:51<00:00,  5.68it/s]



--- Evaluating SRNO_mse_wke_mse (8/8) ---
Weights: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRNO/test_composite_loss_mse_wke_0.6_0.4_multistage/11_09_2026_020543/output/weights/SRNO_best.pth


SRNO_mse_wke_mse (multistage): 100%|██████████| 974/974 [02:50<00:00,  5.70it/s]



=== Done with ['SRNO']. Total results so far: 16 ===


## Aggregate results table (stage-1 alone vs multistage, side by side)

In [20]:
rows = []
for run_label, result in all_results.items():
    row = {
        "Experiment": run_label, "Model": result["model"],
        "Strategy": result["strategy"], "Loss_Norm": result["loss_norm"],
        "epsilon": result["epsilon"],
    }
    for stage_name in ["stage1", "multistage"]:
        metrics = result[stage_name]
        for metric_name, value in metrics.items():
            if set(value.keys()) == {"u", "v"}:
                row[f"{metric_name}_u_{stage_name}"] = value["u"]["mean"]
                row[f"{metric_name}_v_{stage_name}"] = value["v"]["mean"]
            else:
                row[f"{metric_name}_{stage_name}"] = value["mean"]
    rows.append(row)

df_new = pd.DataFrame(rows)

# Merge with whatever's already on disk instead of overwriting it outright --
# this is meant to be run once per architecture across separate sittings, and
# a kernel restart between sittings resets all_results to just the current
# model, so blindly overwriting RESULTS_CSV would silently drop every
# previously-computed architecture. Upsert by Experiment (a fresh recompute
# of the same experiment wins over the stale on-disk row).
if os.path.exists(RESULTS_CSV):
    df_existing = pd.read_csv(RESULTS_CSV)
    df_results = pd.concat([df_existing, df_new]).drop_duplicates(subset="Experiment", keep="last")
else:
    df_results = df_new

df_results = df_results.sort_values(["Model", "Strategy", "Loss_Norm"]).reset_index(drop=True)
df_results.to_csv(RESULTS_CSV, index=False)

# Extra backups, rewritten in full from df_results every time this cell runs
# (so they're always in sync with the merged combined file): one CSV per
# model, and one CSV per individual run_label (a single row each) -- so a
# future bug in the combined file can never destroy more than what it
# actually touches, and any one experiment's numbers can be checked in
# isolation without loading the whole thing.
PER_RUN_DIR = "eval_results_multistage_by_run"
os.makedirs(PER_RUN_DIR, exist_ok=True)

for model in df_results["Model"].unique():
    model_csv = RESULTS_CSV.replace(".csv", f"_{model}.csv")
    df_results[df_results["Model"] == model].to_csv(model_csv, index=False)

for _, row in df_results.iterrows():
    row.to_frame().T.to_csv(os.path.join(PER_RUN_DIR, f"{row['Experiment']}.csv"), index=False)

print(f"\nSaved {len(df_results)} rows to {RESULTS_CSV} ({len(df_new)} from this run)")
print(f"Per-model backups: {sorted(df_results['Model'].unique())}")
print(f"Per-run-label backups: {PER_RUN_DIR}/ ({len(df_results)} files)")
df_results



Saved 32 rows to evaluation_results_multistage_summary.csv (16 from this run)
Per-model backups: ['DSCMS', 'PRUSR', 'SRCNN', 'SRNO']
Per-run-label backups: eval_results_multistage_by_run/ (32 files)


,Experiment,Model,Strategy,Loss_Norm,epsilon,mse_u_stage1,mse_v_stage1,ssim_u_stage1,ssim_v_stage1,psnr_u_stage1,...,lpips_u_multistage,lpips_v_multistage,fsim_u_multistage,fsim_v_multistage,epi_u_multistage,epi_v_multistage,tke_u_multistage,tke_v_multistage,ke_multistage,wke_multistage
0,DSCMS_mse_l1,DSCMS,mse,l1,0.030791,0.000960,0.001142,0.957280,0.957031,38.554043,...,0.193548,0.179306,0.659562,0.729775,0.034209,0.030755,0.017844,0.018045,0.326365,0.428237
1,DSCMS_mse_mse,DSCMS,mse,mse,0.030791,0.000960,0.001142,0.957280,0.957031,38.554043,...,0.179866,0.164510,0.693976,0.738096,0.034563,0.030763,0.015882,0.016241,0.180849,0.224285
2,DSCMS_mse_lpips_l1,DSCMS,mse_lpips,l1,0.017320,0.000293,0.000362,0.987279,0.988529,43.732822,...,0.039164,0.038455,0.825359,0.864891,0.029720,0.025850,0.003893,0.004436,0.023854,0.035954
3,DSCMS_mse_lpips_mse,DSCMS,mse_lpips,mse,0.017320,0.000293,0.000362,0.987279,0.988529,43.732822,...,0.039179,0.038459,0.825153,0.864984,0.029746,0.025854,0.003882,0.004415,0.023968,0.036075
4,DSCMS_mse_lpips_ke_l1,DSCMS,mse_lpips_ke,l1,0.021012,0.000462,0.000514,0.981117,0.984077,41.766624,...,0.057692,0.053260,0.775814,0.831484,0.028931,0.025489,0.007041,0.007228,0.006238,0.008795
5,DSCMS_mse_lpips_ke_mse,DSCMS,mse_lpips_ke,mse,0.021012,0.000462,0.000514,0.981117,0.984077,41.766624,...,0.057681,0.053327,0.775745,0.831488,0.028935,0.025477,0.007032,0.007183,0.008147,0.011605
6,DSCMS_mse_wke_l1,DSCMS,mse_wke,l1,0.052515,0.002918,0.003015,0.918132,0.930001,33.826145,...,0.177312,0.147574,0.640020,0.726819,0.031377,0.028735,0.022607,0.018424,0.106276,0.139331
7,DSCMS_mse_wke_mse,DSCMS,mse_wke,mse,0.052515,0.002918,0.003015,0.918132,0.930001,33.826145,...,0.230804,0.228606,0.586351,0.656644,0.029599,0.027925,0.033088,0.033282,0.090219,0.117396
8,PRUSR_mse_l1,PRUSR,mse,l1,0.027312,0.000793,0.000858,0.961774,0.964787,39.406353,...,0.103795,0.079847,0.768562,0.838121,0.030971,0.026326,0.007414,0.005760,0.067604,0.095909
9,PRUSR_mse_mse,PRUSR,mse,mse,0.027312,0.000793,0.000858,0.961774,0.964787,39.406353,...,0.110150,0.088904,0.750691,0.822759,0.029699,0.026086,0.008166,0.006719,0.054713,0.075818


## Paper-style plots: loss convergence (staged vs single-stage) and averaged spectrum

Mirrors the two figures from the spectrum-informed multistage paper:
- **Loss convergence**: validation loss vs. epoch, colored by phase
  (Stage 1, then Stage 1+Stage 2 continuing from where Stage 1 left off), with a
  single-stage `mse_lpips` run plotted on the same axis as a reference -- directly
  answers "does 2-stage MSE training catch up to single-stage LPIPS training?"
- **Power spectrum**: kinetic-energy spectrum *averaged* over several test
  samples (mean, with a shaded band for the target's spread across samples),
  rather than the single fixed-sample spectrum used above -- closer to the
  paper's aesthetic and more statistically meaningful.

The loss-convergence plot is pure log-parsing (zero GPU cost, safe to run
anytime). The spectrum plot runs real model forward passes (`N_SPECTRUM_SAMPLES`
of them) -- kept small by default to stay light while the training queue is
still running; increase it once the GPU is free.

In [21]:
def find_stage1_log(model_name, strategy, base_dir=BASE_MODEL_DIR):
    """Finds the original (non-multistage) stage-1 training log for a given
    model/strategy, picking the most-recently-modified match if more than one
    exists (mirrors the dedup convention used elsewhere in this project)."""
    model_dir = os.path.join(base_dir, model_name)
    pattern = os.path.join(model_dir, "**", "output", "logs", f"{model_name.upper()}_loss.txt")

    candidates = []
    for path in glob.glob(pattern, recursive=True):
        if "multistage" in path:
            continue
        relative = os.path.relpath(path, model_dir)
        top_folder = relative.split(os.sep)[0]
        if categorize_strategy(top_folder) == strategy:
            candidates.append(path)

    if not candidates:
        return None
    return max(candidates, key=os.path.getmtime)


def parse_stage1_log(path):
    """Stage-1 logs report the composite loss's MSE component separately
    (LOSS_MODE>=1 always includes it), so this is directly comparable to
    multistage's "Final MSE" regardless of stage 1's own LOSS_MODE."""
    text = open(path).read()
    epochs = [int(x) for x in re.findall(r"Epoch \[(\d+)/", text)]
    val_mse = [float(x) for x in re.findall(r"Val\s+-> Total: [0-9.eE+-]+ \| MSE: ([0-9.eE+-]+)", text)]
    return epochs, val_mse


def parse_multistage_log(path):
    text = open(path).read()
    epochs = [int(x) for x in re.findall(r"Epoch \[(\d+)/", text)]
    val_final_mse = [float(x) for x in re.findall(
        r"Val\s+-> Residual loss: [0-9.eE+-]+ \| Final MSE: ([0-9.eE+-]+)", text)]
    return epochs, val_final_mse


def multistage_log_path(weight_path, model_name):
    """Weights live at .../<run>/output/weights/{MODEL}_best.pth; the matching
    log sits at .../<run>/output/logs/{MODEL}_multistage_loss.txt."""
    return (weight_path
            .replace(os.sep + "weights" + os.sep, os.sep + "logs" + os.sep)
            .replace(f"{model_name.upper()}_best.pth", f"{model_name.upper()}_multistage_loss.txt"))


In [22]:
def plot_loss_convergence_paper_style(model_name, strategy, loss_norm="mse", save_path=None):
    """Validation MSE vs. epoch: Stage 1's own curve, then Stage 2 continuing
    from where Stage 1 stopped, plus a single-stage mse_lpips run on the same
    axis as a reference -- same visual language as Figure 3 in the
    spectrum-informed multistage paper (staged vs single-stage convergence).
    Plotted in epochs rather than iterations: stage 1 and stage 2 always use
    the same batch size for a given model, so epochs and iterations differ
    only by a constant factor here -- epochs is the more directly readable
    axis and needs no iters_per_epoch/batch-size bookkeeping."""
    stage1_log = find_stage1_log(model_name, strategy)
    if stage1_log is None:
        print(f"No stage-1 log found for {model_name} {strategy}")
        return

    ms_entry = next(
        (e for e in find_multistage_checkpoints(model_name)
         if e["strategy"] == strategy and e["loss_norm"] == loss_norm),
        None,
    )
    if ms_entry is None:
        print(f"No multistage checkpoint found for {model_name} {strategy} ({loss_norm})")
        return

    s1_epochs, s1_val = parse_stage1_log(stage1_log)
    offset = s1_epochs[-1] if s1_epochs else 0

    ms_log = multistage_log_path(ms_entry["weight_path"], model_name)
    ms_epochs, ms_val = parse_multistage_log(ms_log)
    ms_epochs_shifted = [offset + e for e in ms_epochs]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.semilogy(s1_epochs, s1_val, color="black", lw=1.5, label=f"Stage 1 ({strategy})")
    ax.semilogy(ms_epochs_shifted, ms_val, color="tab:orange", lw=1.5,
                label=f"Stage 1 + Stage 2 ({strategy}, {loss_norm}-trained)")

    if strategy != "mse_lpips":
        lpips_ref_log = find_stage1_log(model_name, "mse_lpips")
        if lpips_ref_log is not None:
            lp_epochs, lp_val = parse_stage1_log(lpips_ref_log)
            ax.semilogy(lp_epochs, lp_val, color="tab:red", lw=1.5, label="Single-stage (mse_lpips)")

    ax.axvline(offset, color="gray", ls=":", lw=1, label="Stage 1 -> Stage 2 handoff")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation MSE (log scale)")
    ax.set_title(f"{model_name} -- Loss convergence: staged vs single-stage")
    ax.grid(True, which="both", ls="--", alpha=0.3)
    ax.legend(fontsize=9)

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


In [23]:
def plot_spectrum_paper_style(model_name, strategy, loss_norm="mse", n_samples=15, save_path=None):
    """Kinetic-energy spectrum averaged over n_samples test-set samples, HR shown
    as a shaded mean +/- std band (the "Target" in the paper's own plot) with
    Stage 1 Alone / Final as plain lines -- same visual language as Figure 1/3's
    left panel. compute_energy_spectrum's k_bins are a fixed function of the
    (always 128x128) grid, so stacking across samples is safe -- same k_bins
    every time, no ragged arrays."""
    ms_entry = next(
        (e for e in find_multistage_checkpoints(model_name)
         if e["strategy"] == strategy and e["loss_norm"] == loss_norm),
        None,
    )
    if ms_entry is None:
        print(f"No multistage checkpoint found for {model_name} {strategy} ({loss_norm})")
        return

    multistage, stage1, stage2, epsilon = load_multistage(model_name, ms_entry["weight_path"])

    n_samples = min(n_samples, len(test_dataset))
    indices = np.linspace(0, len(test_dataset) - 1, n_samples, dtype=int)

    hr_spectra, s1_spectra, final_spectra = [], [], []
    k_bins = None
    with torch.no_grad():
        for idx in indices:
            lr, hr = test_dataset[int(idx)]
            lr, hr = lr.unsqueeze(0).to(device), hr.unsqueeze(0).to(device)
            stage1_pred = stage1(lr)
            final_pred = stage1_pred + epsilon * stage2(lr)

            k_bins, e_hr = compute_energy_spectrum(hr.squeeze(0).cpu().numpy())
            _, e_s1 = compute_energy_spectrum(stage1_pred.squeeze(0).cpu().numpy())
            _, e_final = compute_energy_spectrum(final_pred.squeeze(0).cpu().numpy())
            hr_spectra.append(e_hr)
            s1_spectra.append(e_s1)
            final_spectra.append(e_final)

    hr_arr = np.stack(hr_spectra)
    s1_arr = np.stack(s1_spectra)
    final_arr = np.stack(final_spectra)

    hr_mean, hr_std = hr_arr.mean(axis=0), hr_arr.std(axis=0)
    s1_mean = s1_arr.mean(axis=0)
    final_mean = final_arr.mean(axis=0)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.fill_between(k_bins, np.clip(hr_mean - hr_std, 1e-30, None), hr_mean + hr_std,
                     color="gray", alpha=0.4, label="Target (HR, mean +/- std)")
    ax.loglog(k_bins, hr_mean, color="black", lw=1, alpha=0.6)
    ax.loglog(k_bins, s1_mean, color="tab:purple", lw=2, ls="--", label="Stage 1 Alone")
    ax.loglog(k_bins, final_mean, color="tab:orange", lw=2, label="Stage 1 + Stage 2 (Final)")

    ax.set_xlabel("Wavenumber $k$")
    ax.set_ylabel("$E(k)$")
    ax.set_title(f"{model_name} ({strategy}, {loss_norm}) -- Power spectrum, averaged over {n_samples} samples")
    ax.grid(True, which="both", ls="--", alpha=0.3)
    ax.legend(fontsize=9)

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


In [24]:
# Example usage -- adjust model_name/strategy to whatever's finished so far.
# Loss convergence is free (no GPU); the spectrum plot does n_samples forward
# passes, so keep n_samples small while the training queue is still running.

plot_loss_convergence_paper_style(
    "SRCNN", "mse",
    save_path=os.path.join(PLOTS_DIR, "SRCNN_mse_loss_convergence.png"),
)

plot_spectrum_paper_style(
    "SRCNN", "mse", n_samples=15,
    save_path=os.path.join(PLOTS_DIR, "SRCNN_mse_spectrum_paper_style.png"),
)
